## Dependent and Independent Variables

This code creates the dependent and independent datasets for our project.

In [ ]:
#Go to repo
#%cd /home/jupyter-toomeyck/HelpHerInvest

In [2]:
#Sync latest from GitHub before editing
#!git pull --rebase origin main

In [1]:
# Imports 
import time
import requests
import pandas as pd
#%pip install yfinance --quiet
import yfinance as yf
from pathlib import Path
import numpy as np
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings("ignore")

In [2]:
'''
from pathlib import Path
import subprocess

def get_repo_root(start: Path | None = None) -> Path:
    """
    Return the git repo root (top-level) from anywhere inside the repo.
    Works in notebooks (no __file__ needed).
    """
    start = (start or Path.cwd()).resolve()
    try:
        root = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=start,
            text=True
        ).strip()
        return Path(root)
    except Exception:
        # Fallback: walk upward until we find a .git directory
        for p in [start, *start.parents]:
            if (p / ".git").exists():
                return p
        raise FileNotFoundError("Not inside a git repository (could not find .git).")

REPO_ROOT = get_repo_root()
DATA_DIR = REPO_ROOT / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
'''

'\nfrom pathlib import Path\nimport subprocess\n\ndef get_repo_root(start: Path | None = None) -> Path:\n    """\n    Return the git repo root (top-level) from anywhere inside the repo.\n    Works in notebooks (no __file__ needed).\n    """\n    start = (start or Path.cwd()).resolve()\n    try:\n        root = subprocess.check_output(\n            ["git", "rev-parse", "--show-toplevel"],\n            cwd=start,\n            text=True\n        ).strip()\n        return Path(root)\n    except Exception:\n        # Fallback: walk upward until we find a .git directory\n        for p in [start, *start.parents]:\n            if (p / ".git").exists():\n                return p\n        raise FileNotFoundError("Not inside a git repository (could not find .git).")\n\nREPO_ROOT = get_repo_root()\nDATA_DIR = REPO_ROOT / "Data"\nDATA_DIR.mkdir(parents=True, exist_ok=True)\n\nprint("Repo root:", REPO_ROOT)\nprint("Data dir:", DATA_DIR)\n'

In [3]:
import os
repo_url = "https://github.com/tongyuguo/HelpHerInvest.git"
repo_dir = "HelpHerInvest"
if not os.path.exists(repo_dir):
    !git clone {repo_url}
else:
    !git -C {repo_dir} pull

# list available data files
!ls -lah {repo_dir}/Data

error: Pulling is not possible because you have unmerged files.
hint: Fix them up in the work tree, and then use 'git add/rm <file>'
hint: as appropriate to mark resolution and make a commit.
fatal: Exiting because of an unresolved conflict.
total 16M
drwxr-xr-x  2 jupyter-toomeyck jupyter-toomeyck  162 Feb  4 00:52 .
drwxr-xr-x 11 jupyter-toomeyck jupyter-toomeyck 4.0K Jan 31 00:24 ..
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 2.4M Feb  4 00:50 dependent_variables.csv
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 677K Feb  4 00:50 independent_variables.csv
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 2.9M Feb  4 00:52 production_dataset.csv
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 9.4M Jan 31 03:13 stock_symbols_new.csv.zip
-rw-r--r--  1 jupyter-toomeyck jupyter-toomeyck 296K Jan 30 21:45 testing_small.csv.zip


# X Variables (Independent) Dataset

In [4]:
# X-Variables #
DATA_DIR = Path(repo_dir) / "Data"
## Create Path to Dataset ##
dataset_path = f"{repo_dir}/Data/stock_symbols_new.csv.zip"
csv_file_name = "stock_symbols_new.csv"

# Open the zip file and then open the specific CSV file within it
with zipfile.ZipFile(dataset_path, 'r') as zf:
    with zf.open(csv_file_name) as file_handle:
        # Read the file-like object directly into pandas
        symbols = pd.read_csv(file_handle)

## Reading Symbols from Dataset ##
tickers = symbols['symbol'].tolist()
tickers.remove("SPY")
tickers = tickers[:50]  # limit to 50 for testing
benchmark = 'SPY'
all_tickers = tickers + [benchmark]

## Download Price Data ##
prices = yf.download(
    all_tickers,
    start='2010-01-01',
    auto_adjust=False,
    progress=False
)['Adj Close']

prices = prices.dropna(how='all')

print(prices.head(10))

## Compute Features ##
monthly_px = prices.resample('ME').last()  # month-end prices
mom_1m  = monthly_px / monthly_px.shift(1)  - 1  # 1-month momentum
mom_3m  = monthly_px / monthly_px.shift(3)  - 1  # 3-month momentum
mom_6m  = monthly_px / monthly_px.shift(6)  - 1  # 6-month momentum
mom_12m = monthly_px / monthly_px.shift(12) - 1  # 12-month momentum
mom_12m_ex_1m = (monthly_px.shift(1) / monthly_px.shift(12)) - 1  # 12-month momentum excluding most recent month

rel_3m_spy  = mom_3m.sub(mom_3m["SPY"], axis=0)  # relative strength against S&P 3-month
rel_6m_spy  = mom_6m.sub(mom_6m["SPY"], axis=0)  # relative strength against S&P 6-month
rel_12m_spy = mom_12m.sub(mom_12m["SPY"], axis=0) # relative strength against S&P 12-month

daily_ret = prices.pct_change() # daily returns

vol_3m = (daily_ret.rolling(63).std() * np.sqrt(252)).resample("M").last() # 3-month volatility
vol_6m = (daily_ret.rolling(126).std() * np.sqrt(252)).resample("M").last() # 6-month volatility


roll_max_6m  = monthly_px.rolling(6).max()  # 6-month rolling max
roll_max_12m = monthly_px.rolling(12).max() # 12-month rolling max

drawdown_6m  = monthly_px / roll_max_6m  - 1  # 6-month drawdown
drawdown_12m = monthly_px / roll_max_12m - 1  # 12-month drawdown

dma_200 = prices.rolling(200).mean().resample("M").last() # 200-day moving average
pct_above_200dma = monthly_px / dma_200 - 1  # pct above 200-day moving average


## Combine Features ##
X = pd.concat(
    {
        "mom_1m": mom_1m[tickers],
        "mom_3m": mom_3m[tickers],
        "mom_6m": mom_6m[tickers],
        "mom_12m": mom_12m[tickers],
        "mom_12m_ex_1m": mom_12m_ex_1m[tickers],
        "rel_3m_spy": rel_3m_spy[tickers],
        "rel_6m_spy": rel_6m_spy[tickers],
        "rel_12m_spy": rel_12m_spy[tickers],
        "vol_3m": vol_3m[tickers],
        "vol_6m": vol_6m[tickers],
        #"downside_vol_6m": downside_vol_6m[tickers],
        "drawdown_6m": drawdown_6m[tickers],
        "drawdown_12m": drawdown_12m[tickers],
        "pct_above_200dma": pct_above_200dma[tickers],
    },
    axis=1
)


## Standardize Data Function - z score ##
def zscore_cs(row: pd.Series) -> pd.Series:
    # row contains values across tickers for a single feature at a single date
    mu = row.mean()
    sd = row.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return row * 0.0
    return (row - mu) / sd # calcs z-score


## Normalize per feature across tickers at each date ##
X_z = X.copy()
#print("X_z before normalization:")
#print(X_z.head(20))

for feat in X.columns.get_level_values(0).unique():
    X_z[feat] = X[feat].apply(zscore_cs, axis=1)

#print("X_z after normalization:")
#print(X_z.head(20))

## Flatten X_z table so each  ticker is a row ##
X_panel = (
    X_z.stack(level=1)              # index becomes (Date, Ticker)
      .rename_axis(index=["Date","Ticker"])
      .reset_index()
)


#print(X_panel.head())
print("-----  -----  -----")
print(X_panel.tail())
print("-----  -----  -----")

print("Size of dataset:",
"Rows:",X_panel.shape[0],
"Columns:",X_panel.shape[1])  


output = DATA_DIR / "dependent_variables.csv"
#output = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/dependent_variables.csv"
X_panel.to_csv(output, index=False)
print(f"Output file saved to: {output}")


Ticker          AAPL  ABBV       AMAT   AMD    AMZN       ASML      AVGO  \
Date                                                                       
2010-01-04  6.418383   NaN  11.027144  9.70  6.6950  32.425724  1.328563   
2010-01-05  6.429480   NaN  10.942316  9.71  6.7345  32.678314  1.338425   
2010-01-06  6.327210   NaN  10.919185  9.57  6.6125  32.977695  1.348991   
2010-01-07  6.315511   NaN  10.803514  9.47  6.5000  32.060863  1.340539   
2010-01-08  6.357499   NaN  11.219925  9.43  6.6760  31.293730  1.350401   
2010-01-11  6.301420   NaN  11.466681  9.14  6.5155  30.629494  1.358853   
2010-01-12  6.229739   NaN  10.950028  8.65  6.3675  30.676273  1.338425   
2010-01-13  6.317613   NaN  11.004009  9.15  6.4555  31.527613  1.297568   
2010-01-14  6.281025   NaN  11.065696  9.00  6.3675  31.312441  1.298272   
2010-01-15  6.176056   NaN  10.587597  8.84  6.3570  30.620131  1.284888   

Ticker            AZN  BABA        BAC  ...        RTX        SAP        SPY  \
Date   

## Y Variable (Dependent) Dataset

In [5]:
# Y-Variable #

DATA_DIR = Path(repo_dir) / "Data"
## Create Path to Dataset ##
dataset_path = f"{repo_dir}/Data/stock_symbols_new.csv.zip"
csv_file_name = "stock_symbols_new.csv"

# Open the zip file and then open the specific CSV file within it
with zipfile.ZipFile(dataset_path, 'r') as zf:
    with zf.open(csv_file_name) as file_handle:
        # Read the file-like object directly into pandas
        symbols = pd.read_csv(file_handle)

## Reading Symbols from Dataset ##
tickers = symbols['symbol'].tolist()
tickers.remove("SPY")
tickers = tickers[:50]  # limit to 50 for testing
benchmark = 'SPY'
all_tickers = tickers + [benchmark]

def forward_excess_return_monthly(tickers, benchmark="SPY", start="2010-01-01", end=None, horizon_months=3):

    universe = list(dict.fromkeys(list(tickers) + [benchmark]))

    px_daily = yf.download(
        universe, start=start, end=end, auto_adjust=False, progress=False
    )["Adj Close"]

    px_m = px_daily.resample("M").last().dropna(subset=[benchmark])

    fwd_ret = px_m.shift(-horizon_months) / px_m - 1.0
    bench_fwd = fwd_ret[benchmark].rename("bench_fwd_return")

    # Wide to long in one go
    out = pd.DataFrame(index=px_m.index)
    out["bench_fwd_return"] = bench_fwd

    for t in tickers:
        out[f"adj_close_{t}"] = px_m[t]
        out[f"fwd_return_{t}"] = fwd_ret[t]
        out[f"fwd_excess_{t}"] = fwd_ret[t] - bench_fwd

    return out

df_y_m = forward_excess_return_monthly(all_tickers, benchmark="SPY", start="2010-01-01", horizon_months=6)
print(df_y_m.head(10))
print(df_y_m.tail(10))

bench_df = (
    df_y_m[["bench_fwd_return"]]
    .rename(columns={"bench_fwd_return": "bench_fwd_return"})
    .reset_index()
)

# Keep only stock-level columns
stock_cols = [c for c in df_y_m.columns if "_" in c and not c.startswith("fwd_ret_bench")]

y_long = (
    df_y_m[stock_cols]
    .reset_index()
    .melt(id_vars="Date", var_name="metric_ticker", value_name="value")
)

# Split "adj_close_AAPL" -> metric="adj_close", ticker="AAPL"
y_long[["metric", "Ticker"]] = y_long["metric_ticker"].str.rsplit("_", n=1, expand=True)

y_long = y_long.drop(columns="metric_ticker")

df_y_m_output = (
    y_long
    .pivot(index=["Date", "Ticker"], columns="metric", values="value")
    .reset_index()
)

df_y_m_output = df_y_m_output[["Date", "Ticker", "adj_close", "fwd_excess", "fwd_return"]]
df_y_m_output = df_y_m_output[df_y_m_output["Ticker"] != "return"]

print(df_y_m_output.head(10))
print(df_y_m_output.tail(10))

print("Size of dataset:",
"Rows:",df_y_m_output.shape[0],
"Columns:",df_y_m_output.shape[1])  

# Save to CSV
y_output = DATA_DIR / "independent_variables.csv"
#y_output = "/Users/christoomey/Desktop/Courses/ADAN8888_Applied_Analytics_Project/independent_variables.csv"
df_y_m_output.to_csv(y_output, index=False)
print(f"Output file saved to: {y_output}")


            bench_fwd_return  adj_close_NVDA  fwd_return_NVDA  \
Date                                                            
2010-01-31          0.035952        0.352752        -0.402859   
2010-02-28         -0.040575        0.371318        -0.424074   
2010-03-31         -0.014642        0.398823        -0.328736   
2010-04-30          0.007416        0.360087        -0.234882   
2010-05-31          0.094369        0.301180         0.035769   
2010-06-30          0.231235        0.234022         0.508325   
2010-07-31          0.179371        0.210643         1.602828   
2010-08-31          0.277817        0.213852         1.428725   
2010-09-30          0.172929        0.267716         0.580479   
2010-10-31          0.162489        0.275509         0.663893   

            fwd_excess_NVDA  adj_close_GOOGL  fwd_return_GOOGL  \
Date                                                             
2010-01-31        -0.438811        13.162311         -0.085085   
2010-02-28        -0.

In [6]:
merged_df = pd.merge(
    X_panel,
    df_y_m_output,
    on=["Date", "Ticker"],
    how="inner"
)

print((merged_df.tail(10)))
print("Size of merged dataset:",
"Rows:",merged_df.shape[0],
"Columns:",merged_df.shape[1])

production_dataset_path = DATA_DIR / "production_dataset.csv"
merged_df.to_csv(production_dataset_path, index=False)
print(f"Production dataset saved to: {production_dataset_path}")


           Date Ticker    mom_1m    mom_3m    mom_6m   mom_12m  mom_12m_ex_1m  \
9201 2026-02-28    NVO -4.012734 -0.278003 -0.758926 -1.288853      -1.131574   
9202 2026-02-28    WFC  0.459196  0.050044 -0.226115 -0.299221      -0.330711   
9203 2026-02-28     PM -0.408493  0.395425 -0.338847 -0.327851      -0.295989   
9204 2026-02-28    IBM -1.098308 -0.661687 -0.033345 -0.315374      -0.229386   
9205 2026-02-28  CYATY -0.503983 -0.340673 -0.232363       NaN            NaN   
9206 2026-02-28    SAP -0.538609 -1.468167 -1.130103 -1.046657      -1.029281   
9207 2026-02-28    RTX  0.270728  0.566790  0.129431  0.256212       0.239276   
9208 2026-02-28    MRK  1.241665  0.279831  0.385233 -0.134419      -0.225883   
9209 2026-02-28   AMAT -0.349969  1.151767  1.690202  1.013447       1.076048   
9210 2026-02-28    UNH -0.305184 -1.166728 -0.673532 -1.221292      -1.218974   

      rel_3m_spy  rel_6m_spy  rel_12m_spy    vol_3m    vol_6m  drawdown_6m  \
9201   -0.278003   -0.758926  